# Evaluation pipeline

## Setup envoironment

In [ ]:
import os, sys
os.chdir(os.path.dirname(sys.prefix))
print(f"Working directory: {os.getcwd()}")

## Import libs

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision
from torchvision.transforms import v2

import mlflow.pytorch

import os
import cv2
import json
import mlflow
import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
from src.tps_dewarp.dataset import TPSDataset
from src.models.tpsresnet18 import TPSResNet18
from src.tps_dewarp.transforms import LetterboxResize
from src.tps_dewarp.geometry import build_remap_from_delta_tps

## Setup dugshub, mlflow

In [ ]:
import dagshub
dagshub.init(repo_owner='trxxnk', repo_name='text-image-alignment', mlflow=True)

## Dataset

### Load Dataset

In [ ]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    LetterboxResize(256),
    v2.Normalize(mean=[0.5], std=[0.5]),
])

In [ ]:
DATASET_DIR = "data/generated/v3"
dataset_with_meta = TPSDataset(DATASET_DIR, transform=transform, return_meta=True)
len(dataset_with_meta)

33428

## Eval example

In [ ]:
N = 3337
img, tps, difficulty, meta = dataset_with_meta[N]
meta

{'original': 'raw_18113.png',
 'warped': 'raw_18113_medium.png',
 'is_identity': False}

In [ ]:
img_path = f"data/generated/v3/raw_18113_medium.png"
warped_img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)

map_x, map_y = build_remap_from_delta_tps(
    delta_tps=tps.numpy(),
    H=warped_img.shape[0],
    W=warped_img.shape[1],
    grid_size=9,
    clip=False
)

rectified = cv2.remap(
    warped_img, map_x, map_y,
    interpolation=cv2.INTER_CUBIC,
    borderMode=cv2.BORDER_CONSTANT,
    borderValue=255
)

cv2.imwrite(f'out.png', rectified)

True

In [ ]:
N = 3337
img, tps, difficulty, meta = dataset_with_meta[N]
meta

{'original': 'raw_18113.png',
 'warped': 'raw_18113_medium.png',
 'is_identity': False}

In [ ]:
img = Image.open("data/generated/v3/raw_18113_medium.png").convert("L")

In [ ]:
MLFLOW_MODEL_ID = "your_model_id_here"
model = mlflow.pytorch.load_model(f"models:/{MLFLOW_MODEL_ID}/latest")

In [ ]:
# ---------- INFERENCE ----------
with torch.no_grad():
    pred = model(transform(img).unsqueeze(0)).detach().numpy().round(4).reshape(9*9, 2) 
pred[:5]

array([[0.0025, 0.0001],
       [0.0038, 0.0002],
       [0.0014, 0.0014],
       [0.0022, 0.0019],
       [0.0016, 0.0022]], dtype=float32)

In [ ]:
map_x, map_y = build_remap_from_delta_tps(
    delta_tps=pred,
    H=warped_img.shape[0],
    W=warped_img.shape[1],
    grid_size=9,
    clip=False
)

rectified = cv2.remap(
    warped_img, map_x, map_y,
    interpolation=cv2.INTER_CUBIC,
    borderMode=cv2.BORDER_CONSTANT,
    borderValue=255
)

cv2.imwrite(f'out_model.png', rectified)

True

In [ ]:
tps[:5]

tensor([[0.0000, 0.0000],
        [0.0000, 0.0107],
        [0.0000, 0.0193],
        [0.0000, 0.0241],
        [0.0000, 0.0241]])